![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Consolidator Warm-Up Research

Warm indicators with a consolidator before building indicator and signal series.

### Set Up QuantBook

Create a minute SPY subscription for the consolidator and history request.

In [1]:
qb = QuantBook()
qb.set_start_date(2024, 9, 1)
qb.settings.seed_initial_prices = True
equity = qb.add_equity("SPY", Resolution.MINUTE)

### Build Consolidator Indicators

Create a 30-minute consolidator and indicators for the analysis.

In [2]:
consolidation_minutes = 30
consolidation_period = timedelta(minutes=consolidation_minutes)
# Build a 30-minute consolidator from the minute subscription.
consolidator = TradeBarConsolidator(consolidation_period)
indicator_by_name = {"ema": ExponentialMovingAverage(10), "rsi": RelativeStrengthIndex(12)}
warm_up_period = max(indicator.warm_up_period for indicator in indicator_by_name.values()) * consolidation_minutes
for indicator in indicator_by_name.values():
    qb.register_indicator(equity, indicator, consolidator)

### Build Time Series

Feed minute history into the consolidator and store the 30-minute EMA and RSI values.

In [3]:
indicator_rows = []

def consolidation_handler(self, consolidated_bar: TradeBar):
    if not all([indicator.is_ready for indicator in indicator_by_name.values()]):
        return
    # Store one row per 30-minute bar after both indicators are ready.
    indicator_rows.append(
        {"time": consolidated_bar.end_time, "price": consolidated_bar.close} | 
        {name: indicator.current.value for name, indicator in indicator_by_name.items()}
    )

consolidator.data_consolidated += consolidation_handler
# Feed minute history through the consolidator to build the 30-minute series.
for bar in qb.history[TradeBar](equity, warm_up_period + consolidation_minutes * 15):
    consolidator.update(bar)
indicator_values = pd.DataFrame(indicator_rows).set_index("time")
indicator_values

,price,ema,rsi
time,,,
2024-08-29 15:00:00,546.970956,549.430462,49.674043
2024-08-29 15:30:00,547.587973,549.095464,52.661333
2024-08-29 16:00:00,546.843635,548.686041,48.845645
2024-08-30 10:00:00,550.153981,548.952939,62.151013
2024-08-30 10:30:00,549.713254,549.091178,59.888595
2024-08-30 11:00:00,548.440044,548.972790,53.725175
2024-08-30 11:30:00,547.666324,548.735251,50.293828
2024-08-30 12:00:00,547.617355,548.531997,50.073016
2024-08-30 12:30:00,546.618375,548.184066,45.616009


### Signal Series

Display a Series of 1/0 entry and exit signals generated from the 30-minute indicator values.

In [4]:
signals = ((indicator_values["price"] > indicator_values["ema"]) & (indicator_values["rsi"] > 50)).astype(int)
signals

time
2024-08-29 15:00:00    0
2024-08-29 15:30:00    0
2024-08-29 16:00:00    0
2024-08-30 10:00:00    1
2024-08-30 10:30:00    1
2024-08-30 11:00:00    0
2024-08-30 11:30:00    0
2024-08-30 12:00:00    0
2024-08-30 12:30:00    0
2024-08-30 13:00:00    0
2024-08-30 13:30:00    0
2024-08-30 14:00:00    0
2024-08-30 14:30:00    1
2024-08-30 15:00:00    1
2024-08-30 15:30:00    1
dtype: int64